# 06 — Robust XGBoost Loss & Engine-Weight Experiments

This notebook compares **XGBoost objectives and sample-weight policies** on the pooled NASA C-MAPSS FD001–FD004 problem. It reuses the project decisions supported by prior experiments: a 10-cycle causal rolling window, a 200-cycle capped training target, regime-aware sensor normalization, engine-level splits, and age-matched pseudo-cutoffs.

## Candidates

- `reg:squarederror` — control objective.
- `reg:pseudohubererror` (`huber_slope=10`) — robust alternative for large residuals.
- `reg:absoluteerror` — L1-oriented regression.
- `reg:quantileerror` at quantiles 0.35 and 0.50 — tests whether a conservative lower conditional quantile reduces late-RUL errors.
- For each objective, compare ordinary row weighting with **engine-balanced sample weights**, where each engine contributes equal total training weight.

## Evaluation guardrails

- Split complete training engines into development, validation, and a fresh internal audit, separately within each subset.
- Build pseudo-cutoffs from *observed terminal cycle lengths* in the unlabeled test sensor files. This notebook never reads the official `RUL_FD00x.txt` labels for model selection.
- All sensor history features are causal and calculated within engine.
- Fit regime clustering and normalization statistics only on the training engines allowed for each stage.
- Select the model on validation only; evaluate the selected candidate once on the internal audit; then refit it on all complete training trajectories.
- Report MAE, RMSE, R², standard NASA asymmetric score, bias, late-prediction rate, and within-tolerance rates. No metric is promised in advance.

**Important:** This is an exploratory benchmark project on simulated C-MAPSS data, not certified aviation maintenance software. Official test labels were inspected in earlier project iterations, so their later metrics must be reported transparently as exploratory benchmark results.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from packaging.version import Version
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)

if Version(xgb.__version__) < Version("2.0.0"):
    raise RuntimeError(f"xgboost>=2.0 is required for reg:quantileerror; found {xgb.__version__}.")

RANDOM_STATE = 20261026
SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
COLUMN_NAMES = ["unit_nr", "time_cycles", "setting_1", "setting_2", "setting_3"] + [f"s_{i}" for i in range(1, 22)]
SENSOR_COLS = [f"s_{i}" for i in range(1, 22)]
SETTING_COLS = ["setting_1", "setting_2", "setting_3"]
WINDOW = 10
TARGET_CAP = 200
N_REGIMES = 6
N_CUTOFFS_PER_ENGINE = 3

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CMAPSS_DIR = PROJECT_ROOT / "CMAPSSData"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
for folder in (RAW_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def locate_file(filename):
    for path in (RAW_DIR / filename, CMAPSS_DIR / filename, PROJECT_ROOT / filename):
        if path.exists():
            return path
    raise FileNotFoundError(f"Missing {filename}; checked data/raw/, CMAPSSData/, and project root.")


def load_cmapss_file(filename):
    frame = pd.read_csv(locate_file(filename), sep=r"\s+", header=None, names=COLUMN_NAMES, engine="python")
    frame = frame.dropna(axis=1, how="all")
    if frame.shape[1] != len(COLUMN_NAMES):
        raise ValueError(f"Unexpected schema in {filename}: {frame.shape}")
    return frame


def load_training_data():
    parts = []
    for subset in SUBSETS:
        part = load_cmapss_file(f"train_{subset}.txt")
        part["subset"] = subset
        part["engine_key"] = subset + "_" + part["unit_nr"].astype(int).astype(str).str.zfill(3)
        parts.append(part)
    return pd.concat(parts, ignore_index=True)


def load_test_terminal_ages_only():
    ages = {}
    rows = []
    for subset in SUBSETS:
        part = load_cmapss_file(f"test_{subset}.txt")
        terminal = part.groupby("unit_nr")["time_cycles"].max().astype(int).to_numpy()
        ages[subset] = terminal
        rows.append({"subset": subset, "test_engines": len(terminal), "median_observed_terminal_cycle": float(np.median(terminal))})
    return ages, pd.DataFrame(rows)

train_raw = load_training_data()
observed_test_ages, test_age_summary = load_test_terminal_ages_only()
print(f"XGBoost version: {xgb.__version__}")
print(f"Training rows: {len(train_raw):,} | training engines: {train_raw['engine_key'].nunique():,}")
display(test_age_summary)
print("Official test RUL label files were not loaded by this notebook.")

## 1. Create causal features and a fresh engine-wise split

For each sensor, features include the current measurement, trailing rolling mean/std, and one-cycle change. The regime-aware representation adds six operating-setting clusters and z-scored sensor / rolling-mean features. `engine_key`, `unit_nr`, and `RUL_true` are identifiers/targets, never model inputs.

In [ ]:
def engineer_base_features(frame, window=WINDOW):
    ordered = frame.sort_values(["subset", "unit_nr", "time_cycles"]).reset_index(drop=True).copy()
    required = ["subset", "unit_nr", "engine_key", "time_cycles"] + SETTING_COLS + SENSOR_COLS
    features = ordered[required].copy()
    groups = ordered.groupby(["subset", "unit_nr"], sort=False)
    for sensor in SENSOR_COLS:
        signal = groups[sensor]
        features[f"{sensor}_roll_mean_{window}"] = signal.transform(lambda s: s.rolling(window=window, min_periods=1).mean())
        features[f"{sensor}_roll_std_{window}"] = signal.transform(lambda s: s.rolling(window=window, min_periods=2).std(ddof=0)).fillna(0.0)
        features[f"{sensor}_delta_1"] = signal.diff().fillna(0.0)
    for subset in SUBSETS:
        features[f"subset_{subset}"] = (ordered["subset"] == subset).astype(np.int8)
    final_cycle = ordered.groupby("engine_key")["time_cycles"].max()
    features["RUL_true"] = final_cycle.reindex(ordered["engine_key"]).to_numpy() - ordered["time_cycles"].to_numpy()
    return features


ID_COLS = {"subset", "unit_nr", "engine_key", "RUL_true"}
base = engineer_base_features(train_raw, WINDOW)
BASE_FEATURE_COLUMNS = [c for c in base.columns if c not in ID_COLS]
assert base[BASE_FEATURE_COLUMNS].notna().all().all()
assert not ({"engine_key", "unit_nr", "RUL_true", "subset"} & set(BASE_FEATURE_COLUMNS))

DEV_KEYS, VAL_KEYS, AUDIT_KEYS = [], [], []
for i, subset in enumerate(SUBSETS):
    keys = np.sort(train_raw.loc[train_raw["subset"].eq(subset), "engine_key"].unique())
    dev_keys, remainder = train_test_split(keys, test_size=0.40, random_state=RANDOM_STATE + i)
    val_keys, audit_keys = train_test_split(remainder, test_size=0.50, random_state=RANDOM_STATE + 100 + i)
    DEV_KEYS.extend(dev_keys.tolist())
    VAL_KEYS.extend(val_keys.tolist())
    AUDIT_KEYS.extend(audit_keys.tolist())

assert set(DEV_KEYS).isdisjoint(VAL_KEYS)
assert set(DEV_KEYS).isdisjoint(AUDIT_KEYS)
assert set(VAL_KEYS).isdisjoint(AUDIT_KEYS)

split_summary = []
for subset in SUBSETS:
    all_keys = set(train_raw.loc[train_raw["subset"].eq(subset), "engine_key"].unique())
    split_summary.append({"subset": subset, "development_engines": len(all_keys & set(DEV_KEYS)),
                          "validation_engines": len(all_keys & set(VAL_KEYS)), "audit_engines": len(all_keys & set(AUDIT_KEYS))})
display(pd.DataFrame(split_summary))
print(f"Split totals: development={len(DEV_KEYS)}, validation={len(VAL_KEYS)}, audit={len(AUDIT_KEYS)}")

## 2. Create age-matched pseudo-cutoff validation and audit cases

For each held-out training engine, sample three terminal-cycle ages from the matching subset's *unlabeled test sensor trajectories*. If the sampled age reaches beyond the held-out training engine's life, clip to the last pre-failure cycle. The remaining trajectory supplies a known validation RUL. This mimics the deployment decision point without reading any official test RUL labels.

In [ ]:
def build_age_matched_points(features, engine_keys, age_by_subset, n_per_engine=3, seed=420):
    rng = np.random.default_rng(seed)
    allowed = set(engine_keys)
    records = []
    use = features.loc[features["engine_key"].isin(allowed)]
    for engine_key, group in use.groupby("engine_key", sort=True):
        group = group.sort_values("time_cycles")
        subset = str(group["subset"].iloc[0])
        last_cycle = int(group["time_cycles"].max())
        empirical_ages = np.asarray(age_by_subset[subset], dtype=int)
        for replicate in range(n_per_engine):
            sampled_age = int(rng.choice(empirical_ages))
            cutoff = max(1, min(sampled_age, last_cycle - 1))
            candidates = group.loc[group["time_cycles"].le(cutoff)]
            if candidates.empty:
                continue
            row = candidates.iloc[-1]
            actual_cutoff = int(row["time_cycles"])
            records.append({"base_row_idx": int(row.name), "subset": subset, "engine_key": str(engine_key),
                            "unit_nr": int(row["unit_nr"]), "cutoff_cycle": actual_cutoff,
                            "sampled_test_age": sampled_age, "true_rul_cycles": float(last_cycle - actual_cutoff),
                            "replicate": int(replicate)})
    result = pd.DataFrame(records)
    if result.empty:
        raise ValueError("No pseudo-cutoff cases were generated.")
    return result

val_points = build_age_matched_points(base, VAL_KEYS, observed_test_ages, N_CUTOFFS_PER_ENGINE, seed=RANDOM_STATE + 10)
audit_points = build_age_matched_points(base, AUDIT_KEYS, observed_test_ages, N_CUTOFFS_PER_ENGINE, seed=RANDOM_STATE + 20)
print(f"Validation cases: {len(val_points)} across {val_points.engine_key.nunique()} held-out engines")
print(f"Fresh locked audit cases: {len(audit_points)} across {audit_points.engine_key.nunique()} held-out engines")
display(val_points.groupby("subset").agg(cases=("engine_key", "size"), engines=("engine_key", "nunique"),
                                          median_true_rul=("true_rul_cycles", "median"), max_true_rul=("true_rul_cycles", "max")).reset_index())

## 3. Fit regime-aware feature transforms on permitted engines only

The operating-regime clusters and sensor normalization statistics are fitted on development engines for candidate validation. For the audit, they are refitted on development + validation engines. For the final artifact, they are fitted on all training engines. This prevents audit-engine measurements from influencing their own normalization statistics.

In [ ]:
def make_stats(part, source_columns):
    stats = {}
    for col in source_columns:
        vals = pd.to_numeric(part[col], errors="coerce").to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        stats[col] = (float(np.mean(vals)), max(float(np.std(vals, ddof=0)), 1e-6)) if len(vals) else (0.0, 1.0)
    return stats


def fit_regime_transformer(features, fit_engine_keys, random_state=RANDOM_STATE):
    source_columns = SENSOR_COLS + [f"{s}_roll_mean_{WINDOW}" for s in SENSOR_COLS]
    fit = features.loc[features["engine_key"].isin(set(fit_engine_keys))].copy().reset_index(drop=True)
    if fit.empty:
        raise ValueError("No data supplied to fit operating-regime transformer.")
    scaler = StandardScaler().fit(fit[SETTING_COLS])
    kmeans = KMeans(n_clusters=N_REGIMES, n_init=10, random_state=random_state)
    fit["_regime"] = kmeans.fit_predict(scaler.transform(fit[SETTING_COLS])).astype(int)
    group_stats = {}
    for (subset, regime), group in fit.groupby(["subset", "_regime"], sort=True):
        group_stats[(str(subset), int(regime))] = make_stats(group, source_columns)
    subset_stats = {str(s): make_stats(g, source_columns) for s, g in fit.groupby("subset", sort=True)}
    global_stats = make_stats(fit, source_columns)
    return {"setting_scaler": scaler, "kmeans": kmeans, "group_stats": group_stats,
            "subset_stats": subset_stats, "global_stats": global_stats, "n_regimes": N_REGIMES,
            "source_columns": source_columns, "rolling_window": WINDOW,
            "method": "KMeans operating regimes plus subset/regime sensor z-scores"}


def add_regime_features(features, transformer):
    out = features.copy().reset_index(drop=True)
    regime = transformer["kmeans"].predict(transformer["setting_scaler"].transform(out[SETTING_COLS])).astype(int)
    out["_regime"] = regime
    for r in range(int(transformer["n_regimes"])):
        out[f"operating_regime_{r}"] = (regime == r).astype(np.int8)
    positions = {}
    for pos, key in enumerate(zip(out["subset"].astype(str), regime)):
        positions.setdefault((key[0], int(key[1])), []).append(pos)
    for source_col in transformer["source_columns"]:
        means, stds = np.empty(len(out), dtype=float), np.empty(len(out), dtype=float)
        for key, locs in positions.items():
            stats = transformer["group_stats"].get(key, transformer["subset_stats"].get(key[0], transformer["global_stats"]))
            mean, sd = stats.get(source_col, transformer["global_stats"][source_col])
            means[locs], stds[locs] = mean, max(float(sd), 1e-6)
        out[f"{source_col}_regime_z"] = (out[source_col].to_numpy(dtype=float) - means) / stds
    out.drop(columns=["_regime"], inplace=True)
    return out


def feature_view(fit_engine_keys):
    transformer = fit_regime_transformer(base, fit_engine_keys)
    augmented = add_regime_features(base, transformer)
    regime_columns = [f"operating_regime_{i}" for i in range(N_REGIMES)] + [f"{c}_regime_z" for c in SENSOR_COLS + [f"{s}_roll_mean_{WINDOW}" for s in SENSOR_COLS]]
    columns = BASE_FEATURE_COLUMNS + regime_columns
    if augmented[columns].isna().any().any():
        raise ValueError("Non-finite features after regime transform.")
    return augmented, columns, transformer

DEV_FEATURES, FEATURE_COLUMNS, DEV_TRANSFORMER = feature_view(DEV_KEYS)
print("Fixed feature mode: regime-aware")
print("Model features:", len(FEATURE_COLUMNS))
print("The transform above was fit on development engines only.")

## 4. Define evaluation metrics before fitting candidates

The NASA-style score penalizes late RUL predictions more heavily than early ones. The exponential is numerically capped to avoid overflow. For model selection, we use the mean score per engine within each subset, then average subsets equally; MAE and other diagnostics remain visible so one metric does not hide important failures.

In [ ]:
def capped_target(y, cap=TARGET_CAP):
    y = np.asarray(y, dtype=float)
    return y if cap is None else np.minimum(y, float(cap))


def nasa_penalty_array(y_true, y_pred):
    error = np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float)
    exponent = np.where(error < 0, -error / 13.0, error / 10.0)
    return np.expm1(np.clip(exponent, 0.0, 50.0))


def score_points(points, predictions):
    scored = points[["subset", "engine_key", "unit_nr", "cutoff_cycle", "true_rul_cycles"]].copy().reset_index(drop=True)
    scored["predicted_rul_cycles"] = np.maximum(0.0, np.asarray(predictions, dtype=float))
    scored["error_pred_minus_true"] = scored["predicted_rul_cycles"] - scored["true_rul_cycles"]
    scored["absolute_error"] = scored["error_pred_minus_true"].abs()
    scored["squared_error"] = scored["error_pred_minus_true"] ** 2
    scored["nasa_penalty"] = nasa_penalty_array(scored["true_rul_cycles"], scored["predicted_rul_cycles"])
    scored["within_10_cycles"] = scored["absolute_error"] <= 10
    scored["within_20_cycles"] = scored["absolute_error"] <= 20
    scored["within_30_cycles"] = scored["absolute_error"] <= 30

    # First aggregate repeated cutoffs within each engine, then average engines equally within each subset.
    engine_metrics = scored.groupby(["subset", "engine_key"], as_index=False).agg(
        cases=("engine_key", "size"), mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        nasa_score_per_case=("nasa_penalty", "mean"),
        late_pct=("error_pred_minus_true", lambda x: float(np.mean(x > 0) * 100)),
        mean_bias=("error_pred_minus_true", "mean"),
        within_10_pct=("within_10_cycles", lambda x: float(np.mean(x) * 100)),
        within_20_pct=("within_20_cycles", lambda x: float(np.mean(x) * 100)),
        within_30_pct=("within_30_cycles", lambda x: float(np.mean(x) * 100)),
    )
    by_subset = engine_metrics.groupby("subset", as_index=False).agg(
        engines=("engine_key", "nunique"), cases=("cases", "sum"), mae=("mae", "mean"),
        rmse=("rmse", "mean"), nasa_score_per_case=("nasa_score_per_case", "mean"),
        late_pct=("late_pct", "mean"), mean_bias=("mean_bias", "mean"),
        within_10_pct=("within_10_pct", "mean"), within_20_pct=("within_20_pct", "mean"),
        within_30_pct=("within_30_pct", "mean"),
    )
    summary = {
        "n_cases": int(len(scored)), "n_engines": int(scored["engine_key"].nunique()),
        "pooled_mae": float(scored["absolute_error"].mean()),
        "pooled_rmse": float(np.sqrt(scored["squared_error"].mean())),
        "pooled_r2": float(r2_score(scored["true_rul_cycles"], scored["predicted_rul_cycles"])),
        "macro_subset_mae": float(by_subset["mae"].mean()),
        "macro_subset_rmse": float(by_subset["rmse"].mean()),
        "macro_subset_nasa_score_per_case": float(by_subset["nasa_score_per_case"].mean()),
        "macro_subset_late_pct": float(by_subset["late_pct"].mean()),
        "macro_subset_mean_bias": float(by_subset["mean_bias"].mean()),
        "macro_subset_within_10_pct": float(by_subset["within_10_pct"].mean()),
        "macro_subset_within_20_pct": float(by_subset["within_20_pct"].mean()),
        "macro_subset_within_30_pct": float(by_subset["within_30_pct"].mean()),
    }
    return summary, by_subset, scored, engine_metrics


def sample_weights(training_frame, policy):
    if policy == "row_uniform":
        return None
    if policy != "engine_balanced":
        raise ValueError(f"Unknown sample-weight policy: {policy}")
    per_engine_rows = training_frame.groupby("engine_key")["engine_key"].transform("size").to_numpy(dtype=float)
    weights = 1.0 / np.maximum(per_engine_rows, 1.0)
    return weights / weights.mean()

FIXED_XGB_PARAMS = {
    "n_estimators": 550,
    "max_depth": 5,
    "learning_rate": 0.03,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "min_child_weight": 5,
    "reg_lambda": 7.0,
    "tree_method": "hist",
    "n_jobs": 4,
}

LOSS_CANDIDATES = [
    {"loss_id": "squarederror", "objective": "reg:squarederror", "objective_params": {}},
    {"loss_id": "pseudohuber_slope10", "objective": "reg:pseudohubererror", "objective_params": {"huber_slope": 10.0}},
    {"loss_id": "absoluteerror", "objective": "reg:absoluteerror", "objective_params": {}},
    {"loss_id": "quantile_q035", "objective": "reg:quantileerror", "objective_params": {"quantile_alpha": 0.35}},
    {"loss_id": "quantile_q050", "objective": "reg:quantileerror", "objective_params": {"quantile_alpha": 0.50}},
]
WEIGHT_POLICIES = ["row_uniform", "engine_balanced"]

def weighted_quantile(values, quantile, weights=None):
    values = np.asarray(values, dtype=float)
    if weights is None:
        return float(np.quantile(values, quantile))
    weights = np.asarray(weights, dtype=float)
    order = np.argsort(values)
    sorted_values = values[order]
    sorted_weights = weights[order]
    cumulative = np.cumsum(sorted_weights)
    cutoff = float(quantile) * float(cumulative[-1])
    pos = min(int(np.searchsorted(cumulative, cutoff, side="left")), len(sorted_values) - 1)
    return float(sorted_values[pos])


def initial_base_score(candidate, y_fit, weights):
    """Set a loss-appropriate starting level; needed for stable robust-loss initialization."""
    objective = candidate["objective"]
    if objective == "reg:squarederror":
        return float(np.average(y_fit, weights=weights)) if weights is not None else float(np.mean(y_fit))
    if objective in {"reg:pseudohubererror", "reg:absoluteerror"}:
        return weighted_quantile(y_fit, 0.50, weights)
    if objective == "reg:quantileerror":
        q = float(candidate["objective_params"].get("quantile_alpha", 0.50))
        return weighted_quantile(y_fit, q, weights)
    raise ValueError(f"Unsupported objective in this experiment: {objective}")


def make_model(candidate, y_fit, weights=None, seed=RANDOM_STATE):
    params = dict(FIXED_XGB_PARAMS)
    params.update(candidate["objective_params"])
    params["objective"] = candidate["objective"]
    params["random_state"] = int(seed)
    # Explicit loss-appropriate intercept avoids unstable automatic initialization
    # for robust losses on RUL values measured in hundreds of cycles.
    params["base_score"] = initial_base_score(candidate, y_fit, weights)
    return XGBRegressor(**params)


def fit_candidate(candidate, fit_frame, feature_columns, fit_engine_keys, seed=RANDOM_STATE):
    mask = fit_frame["engine_key"].isin(set(fit_engine_keys))
    fit_rows = fit_frame.loc[mask]
    X_fit = fit_rows[feature_columns]
    y_fit = capped_target(fit_rows["RUL_true"].to_numpy(dtype=float), TARGET_CAP)
    w = sample_weights(fit_rows, candidate["weight_policy"])
    model = make_model(candidate, y_fit, weights=w, seed=seed)
    if w is None:
        model.fit(X_fit, y_fit)
    else:
        model.fit(X_fit, y_fit, sample_weight=w)
    return model


## 5. Compare five objectives × two weighting policies

The feature pipeline, 10-cycle history window, 200-cycle target cap, XGBoost tree settings, split and validation pseudo-cutoffs are held fixed. This isolates the effect of the objective and sample-weight policy. Each objective receives an explicit loss-appropriate base score (weighted mean, median, or quantile) to stabilize initialization on RUL values measured in cycles. Quantile `q=0.35` is a deliberate conservative-RUL candidate because late predictions are more costly under the selected NASA-style score; it may increase early predictions, so the full metric table still determines the choice.

In [ ]:
DEV_MASK = base["engine_key"].isin(set(DEV_KEYS))
X_validation = DEV_FEATURES.iloc[val_points["base_row_idx"].to_numpy(dtype=int)][FEATURE_COLUMNS]

candidate_specs = []
for loss in LOSS_CANDIDATES:
    for weight_policy in WEIGHT_POLICIES:
        candidate_specs.append({**loss, "weight_policy": weight_policy,
                                "candidate_id": f"{loss['loss_id']}__{weight_policy}"})

comparison_rows = []
all_validation_predictions = []
all_validation_by_subset = []

for idx, candidate in enumerate(candidate_specs, start=1):
    model = fit_candidate(candidate, DEV_FEATURES, FEATURE_COLUMNS, DEV_KEYS, seed=RANDOM_STATE)
    predictions = np.maximum(0.0, model.predict(X_validation))
    summary, by_subset, scored, engine_metrics = score_points(val_points, predictions)
    row = {"candidate_id": candidate["candidate_id"], "loss_id": candidate["loss_id"],
           "objective": candidate["objective"], "objective_params": json.dumps(candidate["objective_params"], sort_keys=True),
           "weight_policy": candidate["weight_policy"], "target_cap": TARGET_CAP,
           "rolling_window": WINDOW, "feature_mode": "regime_aware", **summary}
    comparison_rows.append(row)
    all_validation_predictions.append(scored.assign(candidate_id=candidate["candidate_id"], loss_id=candidate["loss_id"], weight_policy=candidate["weight_policy"]))
    all_validation_by_subset.append(by_subset.assign(candidate_id=candidate["candidate_id"], loss_id=candidate["loss_id"], weight_policy=candidate["weight_policy"]))
    print(f"[{idx}/{len(candidate_specs)}] {candidate['candidate_id']}: macro NASA={summary['macro_subset_nasa_score_per_case']:.3f}; macro MAE={summary['macro_subset_mae']:.2f}; late={summary['macro_subset_late_pct']:.1f}%")

comparison = pd.DataFrame(comparison_rows).sort_values(
    ["macro_subset_nasa_score_per_case", "macro_subset_mae", "macro_subset_rmse", "macro_subset_late_pct"], ascending=True
).reset_index(drop=True)
val_predictions_all = pd.concat(all_validation_predictions, ignore_index=True)
val_by_subset_all = pd.concat(all_validation_by_subset, ignore_index=True)
comparison.to_csv(REPORTS_DIR / "loss_objective_validation_comparison.csv", index=False)
val_predictions_all.to_csv(REPORTS_DIR / "loss_objective_validation_predictions.csv", index=False)
val_by_subset_all.to_csv(REPORTS_DIR / "loss_objective_validation_by_subset.csv", index=False)
display(comparison.round(3))

WINNER_ID = str(comparison.iloc[0]["candidate_id"])
WINNER = next(c for c in candidate_specs if c["candidate_id"] == WINNER_ID)
print("Validation-selected candidate:", WINNER_ID)
print("Selection rule: lowest macro-subset NASA asymmetric score per case; tie-break by macro-subset MAE, RMSE, then late-prediction rate.")
print("Locked audit was not used for candidate selection.")

## 6. Diagnose per-subset validation performance and seed stability

This cell visualizes per-subset metrics for the selected candidate and retrains only that fixed candidate with three seeds. Seed sensitivity is descriptive and does not change the selected objective/weight policy.

In [ ]:
winner_rows = val_by_subset_all.loc[val_by_subset_all["candidate_id"].eq(WINNER_ID)].copy()
display(winner_rows.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(winner_rows["subset"], winner_rows["mae"])
axes[0].set(title=f"Validation MAE by subset — {WINNER_ID}", xlabel="Subset", ylabel="MAE (cycles)")
val_winner_predictions = val_predictions_all.loc[val_predictions_all["candidate_id"].eq(WINNER_ID)].copy()
axes[1].scatter(val_winner_predictions["true_rul_cycles"], val_winner_predictions["predicted_rul_cycles"], alpha=0.65)
limit = float(max(val_winner_predictions["true_rul_cycles"].max(), val_winner_predictions["predicted_rul_cycles"].max(), 1))
axes[1].plot([0, limit], [0, limit], linestyle="--", color="gray")
axes[1].set(title="Validation: predicted vs true RUL", xlabel="True RUL (cycles)", ylabel="Predicted RUL (cycles)")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "loss_objective_validation_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

seed_rows = []
for seed in [17, 42, 2026]:
    seed_model = fit_candidate(WINNER, DEV_FEATURES, FEATURE_COLUMNS, DEV_KEYS, seed=seed)
    seed_predictions = np.maximum(0.0, seed_model.predict(X_validation))
    seed_metrics, _, _, _ = score_points(val_points, seed_predictions)
    seed_rows.append({"candidate_id": WINNER_ID, "seed": seed, **seed_metrics})
seed_stability = pd.DataFrame(seed_rows)
seed_stability.to_csv(REPORTS_DIR / "loss_objective_seed_stability.csv", index=False)
display(seed_stability.round(3))

## 7. Evaluate the selected candidate once on the fresh locked audit

The audit engines are not used to choose between objectives or weight policies. We fit the selected configuration on development + validation engines, fit the regime transformer on those engines only, then measure performance on the separate audit engines.

In [ ]:
FIT_KEYS_AUDIT = sorted(set(DEV_KEYS + VAL_KEYS))
AUDIT_FEATURES, AUDIT_FEATURE_COLUMNS, AUDIT_TRANSFORMER = feature_view(FIT_KEYS_AUDIT)
assert AUDIT_FEATURE_COLUMNS == FEATURE_COLUMNS

audit_model = fit_candidate(WINNER, AUDIT_FEATURES, FEATURE_COLUMNS, FIT_KEYS_AUDIT, seed=RANDOM_STATE)
X_audit = AUDIT_FEATURES.iloc[audit_points["base_row_idx"].to_numpy(dtype=int)][FEATURE_COLUMNS]
audit_predictions = np.maximum(0.0, audit_model.predict(X_audit))
audit_summary_metrics, audit_by_subset, audit_scored, audit_engine_metrics = score_points(audit_points, audit_predictions)
audit_scored.to_csv(REPORTS_DIR / "loss_objective_locked_audit_predictions.csv", index=False)
audit_by_subset.to_csv(REPORTS_DIR / "loss_objective_locked_audit_by_subset.csv", index=False)

# Stratified bootstrap resamples whole engines within each subset, not individual cutoff rows.
rng = np.random.default_rng(RANDOM_STATE + 900)
bootstrap_macro_engine_mae = []
for _ in range(2000):
    subset_means = []
    for subset in SUBSETS:
        vals = audit_engine_metrics.loc[audit_engine_metrics["subset"].eq(subset), "mae"].to_numpy(dtype=float)
        if len(vals):
            subset_means.append(float(rng.choice(vals, size=len(vals), replace=True).mean()))
    if subset_means:
        bootstrap_macro_engine_mae.append(float(np.mean(subset_means)))
mae_ci = np.quantile(bootstrap_macro_engine_mae, [0.025, 0.975])

audit_report = {
    "selected_candidate": WINNER_ID,
    "objective": WINNER["objective"],
    "objective_params": WINNER["objective_params"],
    "sample_weight_policy": WINNER["weight_policy"],
    "fixed_xgboost_params": FIXED_XGB_PARAMS,
    "target_cap_cycles": TARGET_CAP,
    "rolling_window_cycles": WINDOW,
    "feature_mode": "regime_aware",
    "feature_count": len(FEATURE_COLUMNS),
    "audit_engines": int(audit_engine_metrics["engine_key"].nunique()),
    "audit_cases": int(len(audit_scored)),
    "metrics": audit_summary_metrics,
    "stratified_engine_bootstrap_95pct_interval_macro_subset_engine_mae": [float(mae_ci[0]), float(mae_ci[1])],
    "per_subset_metrics": audit_by_subset.to_dict(orient="records"),
    "guardrails": {"official_rul_label_files_loaded": False, "audit_used_for_candidate_selection": False,
                   "engine_level_split": True, "transformer_fit_only_on_development_plus_validation_engines": True},
    "interpretation": "Fresh internal age-matched pseudo-cutoff audit on training engines; not official NASA test performance or real-world industrial validation."
}
(REPORTS_DIR / "loss_objective_locked_audit_summary.json").write_text(json.dumps(audit_report, indent=2), encoding="utf-8")
print(json.dumps(audit_report, indent=2))
display(audit_by_subset.round(3))

## 8. Refit the selected objective on all complete training trajectories and save a compatible artifact

Only after objective/weight selection and the audit summary have been produced do we refit on all labelled training engines. The saved dictionary keeps the keys consumed by `app.py` and Notebook 03. A timestamped snapshot of the active model is kept before replacement.

In [ ]:
from datetime import datetime

MODEL_PATH = MODELS_DIR / "xgb_rul_model_all_subsets.joblib"
backup_path = None
if MODEL_PATH.exists():
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = MODELS_DIR / f"xgb_rul_model_pre_loss_experiments_{stamp}.joblib"
    shutil.copy2(MODEL_PATH, backup_path)
    print("Previous model snapshot:", backup_path)

ALL_ENGINE_KEYS = sorted(base["engine_key"].unique().tolist())
FINAL_FEATURES, FINAL_FEATURE_COLUMNS, FINAL_TRANSFORMER = feature_view(ALL_ENGINE_KEYS)
assert FINAL_FEATURE_COLUMNS == FEATURE_COLUMNS
final_rows = FINAL_FEATURES.loc[FINAL_FEATURES["engine_key"].isin(ALL_ENGINE_KEYS)]
X_all = final_rows[FINAL_FEATURE_COLUMNS]
y_all = capped_target(final_rows["RUL_true"].to_numpy(dtype=float), TARGET_CAP)
final_weights = sample_weights(final_rows, WINNER["weight_policy"])
final_model = make_model(WINNER, y_all, weights=final_weights, seed=RANDOM_STATE)
if final_weights is None:
    final_model.fit(X_all, y_all)
else:
    final_model.fit(X_all, y_all, sample_weight=final_weights)

artifact = {
    "model": final_model,
    "sensor_cols": SENSOR_COLS,
    "setting_cols": SETTING_COLS,
    "feature_columns": FINAL_FEATURE_COLUMNS,
    "base_feature_columns": BASE_FEATURE_COLUMNS,
    "rolling_window": int(WINDOW),
    "rul_cap": int(TARGET_CAP),
    "best_params": {**FIXED_XGB_PARAMS, "objective": WINNER["objective"], **WINNER["objective_params"]},
    "dataset": "NASA PCoE C-MAPSS FD001-FD004 pooled XGBoost RUL model",
    "subsets": SUBSETS,
    "target_unit": "operating cycles",
    "random_state": RANDOM_STATE,
    "n_training_rows": int(len(train_raw)),
    "n_training_engines": int(train_raw["engine_key"].nunique()),
    "n_model_features": int(len(FINAL_FEATURE_COLUMNS)),
    "feature_mode": "regime_aware",
    "regime_transformer": FINAL_TRANSFORMER,
    "selected_objective": WINNER["objective"],
    "objective_id": WINNER["loss_id"],
    "objective_specific_params": WINNER["objective_params"],
    "sample_weight_policy": WINNER["weight_policy"],
    "experiment_selection": {
        "candidate_id": WINNER_ID,
        "selection_metric": "validation macro-subset NASA asymmetric score per case, then macro-subset MAE/RMSE and late-prediction rate",
        "validation_engine_split_seed": RANDOM_STATE,
        "locked_audit_used_for_selection": False,
        "official_test_rul_labels_loaded_in_notebook06": False,
        "note": "Official test labels were examined earlier in project development; subsequent official-test results are exploratory benchmark comparisons, not a pristine blind test."
    },
    "sample_weight_policy_description": "row_uniform = equal row weights; engine_balanced = each training engine contributes equal total weight, normalized to mean 1."
}
joblib.dump(artifact, MODEL_PATH)

summary = {
    "model_artifact": str(MODEL_PATH),
    "previous_model_snapshot": str(backup_path) if backup_path else None,
    "dataset": artifact["dataset"],
    "training_rows": artifact["n_training_rows"],
    "training_engines": artifact["n_training_engines"],
    "feature_count": artifact["n_model_features"],
    "selected_candidate": WINNER_ID,
    "selected_objective": WINNER["objective"],
    "objective_specific_params": WINNER["objective_params"],
    "sample_weight_policy": WINNER["weight_policy"],
    "target_cap": TARGET_CAP,
    "rolling_window": WINDOW,
    "feature_mode": "regime_aware",
    "validation_metrics": comparison.loc[comparison["candidate_id"].eq(WINNER_ID)].iloc[0].to_dict(),
    "locked_audit_summary": str(REPORTS_DIR / "loss_objective_locked_audit_summary.json"),
    "official_test_rul_labels_used_for_selection": False,
}
(REPORTS_DIR / "loss_objective_final_training_summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
print("\nFINAL MODEL SAVED:", MODEL_PATH)
print("Selected candidate:", WINNER_ID)
print("Objective:", WINNER["objective"], WINNER["objective_params"])
print("Sample weighting:", WINNER["weight_policy"])
print("Feature count:", len(FINAL_FEATURE_COLUMNS))
print("Next: run notebooks/03_test_evaluation_and_explainability.ipynb once for exploratory official benchmark reporting, then restart Streamlit.")

## 9. Interview interpretation checklist

- Why a 10-cycle rolling window and 200-cycle training cap are fixed for this objective-only ablation.
- Why engine-balanced weights may reduce domination by longer training trajectories.
- What Pseudo-Huber, absolute-error, and quantile objectives optimize, and what they trade off.
- Why a lower quantile can reduce late predictions while increasing early predictions.
- Why the primary score is macro-subset asymmetric penalty and why MAE/RMSE/late rate remain essential diagnostics.
- Why the locked audit is separate from candidate selection and why official test RUL files are never read in this notebook.
- Why benchmark results from simulated NASA C-MAPSS do not establish production safety or readiness for real aircraft maintenance.